In [1]:
%load_ext autoreload
%autoreload 2

import os
from functools import partial

import numpy as np
import pandas as pd

os.environ['OPENAI_KEY_SAE'] = '...' # Replace with your OpenAI API key, or with another environment variable (e.g. os.environ['OPENAI_API_KEY'])
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8010/v1'  # Optional: local OpenAI-compatible endpoint

from hypothesaes.quickstart import train_sae
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.sae import get_multiple_sae_activations
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import NeuronInterpreter, InterpretConfig, ScoringConfig, SamplingConfig, sample_percentile_bins, ternary_sample_percentile_bins, ternary_sample_topn
from hypothesaes.evaluation import score_hypotheses
from hypothesaes.annotate import annotate_texts_with_concepts


In [2]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
base_dir = os.path.join(prefix, "data")

EMBEDDER = "nomic-ai/modernbert-embed-base"
ANNOTATOR_MODEL = "openai/gpt-oss-20b"
INTERPRETER_MODEL = "openai/gpt-oss-120b"
INTERPRET_PROMPT_NAME = "interpret-neuron-binary"

USE_ABS_TOPK = True
USE_VALIDATION = True
PERCENTILE_SAMPLE = False
DATASET_NAME = "headlines"

EMBED_CACHE_NAME = f"{DATASET_NAME}_{EMBEDDER}"
ACT_CACHE_NAME = EMBED_CACHE_NAME
if USE_ABS_TOPK:
    ACT_CACHE_NAME += "_abstopk"
    INTERPRET_PROMPT_NAME = "interpret-neuron-ternary"

INTERP_CACHE_NAME = ACT_CACHE_NAME
if not PERCENTILE_SAMPLE:
    INTERP_CACHE_NAME += "_topnsample"

TRAIN_ANN_CACHE_NAME = INTERP_CACHE_NAME

ANN_CACHE_NAME = INTERP_CACHE_NAME
if USE_VALIDATION:
    ANN_CACHE_NAME += "_validation"

In [3]:
print(
    "EMBED_CACHE_NAME =", EMBED_CACHE_NAME, "\n",
    "ACT_CACHE_NAME =", ACT_CACHE_NAME, "\n",
    "INTERP_CACHE_NAME =", INTERP_CACHE_NAME, "\n",
    "ANN_CACHE_NAME =", ANN_CACHE_NAME, "\n",
    "INTERPRET_PROMPT_NAME =", INTERPRET_PROMPT_NAME,
)

EMBED_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base 
 ACT_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base_abstopk 
 INTERP_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base_abstopk_topnsample 
 ANN_CACHE_NAME = headlines_nomic-ai/modernbert-embed-base_abstopk_topnsample_validation 
 INTERPRET_PROMPT_NAME = interpret-neuron-ternary


In [4]:
label_col = "label_pairwise"
train_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "train-8.8K.json"), lines=True)
val_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "val-1.0K.json"), lines=True)

train_labels = train_df[label_col].values

unique_train_texts = np.unique(train_df["headline_A"].tolist() + train_df["headline_B"].tolist()).tolist()
unique_val_texts = np.unique(val_df["headline_A"].tolist() + val_df["headline_B"].tolist()).tolist()

text2embedding = get_local_embeddings(unique_train_texts + unique_val_texts, model=EMBEDDER, cache_name=EMBED_CACHE_NAME)

train_embeddings = np.stack([text2embedding[text] for text in unique_train_texts])
val_embeddings = np.stack([text2embedding[text] for text in unique_val_texts])

Loading embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 16468 embeddings in 0.0s


In [5]:
checkpoint_dir = os.path.join(prefix, f"checkpoints/{ACT_CACHE_NAME}")
sae_256_8 = train_sae(embeddings=train_embeddings, M=256, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5, abs_topk=USE_ABS_TOPK)
sae_32_4 = train_sae(embeddings=train_embeddings, M=32, K=4, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5, abs_topk=USE_ABS_TOPK)

sae_list = [sae_256_8, sae_32_4]

unique_train_activations, neuron_source_sae_info = get_multiple_sae_activations(sae_list, train_embeddings, return_neuron_source_info=True)


Loaded model from ../checkpoints/headlines_nomic-ai/modernbert-embed-base_abstopk/SAE_M=256_K=8.pt onto device cuda
Loaded model from ../checkpoints/headlines_nomic-ai/modernbert-embed-base_abstopk/SAE_M=32_K=4.pt onto device cuda


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
def get_delta_acts_from_df(df):
    headline_A = np.stack([text2embedding[text] for text in df["headline_A"].tolist()])
    headline_B = np.stack([text2embedding[text] for text in df["headline_B"].tolist()])
    Z_A = get_multiple_sae_activations(sae_list, headline_A)
    Z_B = get_multiple_sae_activations(sae_list, headline_B)
    return Z_A - Z_B

In [8]:
train_Z_delta = get_delta_acts_from_df(train_df)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
selection_cache_dir = os.path.join(prefix, "selection_cache", ACT_CACHE_NAME)
os.makedirs(selection_cache_dir, exist_ok=True)
selected_neurons, scores = select_neurons(
    activations=train_Z_delta,
    target=train_labels,
    n_select=20,
    method="lasso",
    classification=True,
    verbose=True,
    cache_dir=selection_cache_dir
)

In [10]:
import importlib
import hypothesaes.interpret_neurons
importlib.reload(hypothesaes.interpret_neurons)
from hypothesaes.interpret_neurons import NeuronInterpreter, ternary_sample_percentile_bins, ternary_sample_topn

In [11]:
interpreter = NeuronInterpreter(
    interpreter_model=INTERPRETER_MODEL,
    annotator_model=ANNOTATOR_MODEL,
    n_workers_interpretation=4,
    n_workers_annotation=100,
    cache_name=INTERP_CACHE_NAME,
)

n_examples = 20
high_percentile = (80, 100)
n_candidates = 5
annotation_kwargs = {}

if USE_ABS_TOPK:
    if PERCENTILE_SAMPLE:
        interpretation_sampling_config = SamplingConfig(
            function=ternary_sample_percentile_bins,
            n_examples=n_examples,
            sampling_kwargs={
                "high_abs_percentile": high_percentile
            },
        )
    else:
        interpretation_sampling_config = SamplingConfig(
            function=ternary_sample_topn,
            n_examples=n_examples,
        )
else:
    interpretation_sampling_config = SamplingConfig(
        function=sample_percentile_bins,
        n_examples=n_examples,
        sampling_kwargs={
            "high_percentile": high_percentile
        },
    )

interpret_config = InterpretConfig(
    sampling=interpretation_sampling_config,
    n_candidates=n_candidates,
    interpretation_prompt_name=INTERPRET_PROMPT_NAME,
    task_specific_instructions="""All of the texts are digital news headlines.
Features should describe specific characteristics in the headline texts. Example features may include:
- "explicitly mentions climate change or another environmental issue"
- "asks the reader a rhetorical question\"""",
)

In [12]:
interpretations = interpreter.interpret_neurons(
    texts=unique_train_texts,
    activations=unique_train_activations,
    neuron_indices=selected_neurons,
    config=interpret_config,
)

In [13]:
scoring_config = ScoringConfig(
    n_examples=200,
    sampling_function=interpretation_sampling_config.function,
    sampling_kwargs=interpretation_sampling_config.sampling_kwargs
)
score_func = interpreter.ternary_score_interpretations if USE_ABS_TOPK else interpreter.score_interpretations

In [14]:
all_metrics = score_func(
    texts=unique_train_texts,
    activations=unique_train_activations,
    interpretations=interpretations,
    config=scoring_config,
)

[WARNING] Only 62 < 66 negative examples available for 140
[WARNING] Only 62 < 66 neutral examples available for 140
[WARNING] Only 13 < 66 positive examples available for 62
[WARNING] Only 31 < 66 negative examples available for 210
[WARNING] Only 31 < 66 neutral examples available for 210
[WARNING] Only 7 < 66 negative examples available for 243
[WARNING] Only 7 < 66 neutral examples available for 243
[WARNING] Only 27 < 66 positive examples available for 65
[WARNING] Only 27 < 66 positive examples available for 81
[WARNING] Only 34 < 66 negative examples available for 286
[WARNING] Only 34 < 66 neutral examples available for 286
Found 18495 cached items; annotating 0 uncached items


/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/blue/bonniejdorr/ethanlmines/repos/HypotheSAEs/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control 

In [15]:
results = []
for neuron_idx, fidelity_scores in all_metrics.items():
    best_interp = max(interpretations[neuron_idx], key=lambda interp: fidelity_scores[interp]['f1'])
    results.append({
        'neuron_idx': neuron_idx,
        'source_sae': neuron_source_sae_info[neuron_idx],
        'target_lasso_coef': scores[selected_neurons.index(neuron_idx)],
        'interpretation': best_interp,
        'f1_fidelity_score': fidelity_scores[best_interp]['f1'],
    })

In [16]:
hypothesis_df = pd.DataFrame(results).set_index('neuron_idx')
pd.set_option('display.max_colwidth', None)
display(hypothesis_df.sort_values(by='target_lasso_coef', ascending=False))
pd.reset_option('display.max_colwidth')

,source_sae,target_lasso_coef,interpretation,f1_fidelity_score
neuron_idx,,,,
257,"(SAE_1, 32, 4)",0.158348,Mentions a video (positive) or mentions a book/novel/fictional work (negative),0.758338
262,"(SAE_1, 32, 4)",0.109705,Mentions a marginalized identity or social injustice (positive) or mentions an environmental/climate issue (negative),0.668946
140,"(SAE_0, 256, 8)",0.069642,"Mentions photographic media (photos/images) in positive samples versus mentions video/broadcast media (videos, TV, watching) in negative samples",0.612069
41,"(SAE_0, 256, 8)",0.050278,"Contains a strongly valenced adjective: negative adjectives (e.g., “Worst”, “Terrible”, “Disturbing”) in positive samples; positive adjectives (e.g., “Funny”, “Beautiful”, “Well‑Connected”) in negative samples",0.517170
46,"(SAE_0, 256, 8)",0.039174,"Contains a number written as a word (e.g., “One”) in positive samples or contains a numeric digit (e.g., “7”, “49”) in negative samples",0.453088
274,"(SAE_1, 32, 4)",0.020018,"Mentions a speech‑related term such as “Speech” or “TED Talk” (positive) or mentions a visual‑media term such as “Oscars”, “movie”, or “Tabloid” (negative)",0.656542
139,"(SAE_0, 256, 8)",0.017634,"References a protected demographic group (e.g., race in positive samples, gender in negative samples)",0.402299
260,"(SAE_1, 32, 4)",0.016116,"Contains a numeric token (e.g., 51, 5, 50+)",0.514337
62,"(SAE_0, 256, 8)",0.015230,"Mentions a controversial social issue (e.g., LGBTQ+ rights, consent, sexual violence) in positive samples or mentions a controversial environmental/energy issue (e.g., oil spills, pipelines) in negative samples",0.671395


In [17]:
def show_examples(neuron_idx):
    n_ex = 5
    print(hypothesis_df.loc[neuron_idx].interpretation)
    neuron_acts = unique_train_activations[:, neuron_idx]
    sort_inds = np.argsort(neuron_acts)
    def print_example(ind):
        print(f"{neuron_acts[ind]:.4f} : {unique_train_texts[ind]}")
    print("POSITIVE")
    print("-----------------")
    print("Activation : Text")
    print("-----------------")
    for i in range(-1, -n_ex-1, -1):
        print_example(sort_inds[i])
    print()
    print("NEGATIVE")
    print("-----------------")
    print("Activation : Text")
    print("-----------------")
    for i in range(n_ex):
        print_example(sort_inds[i])
# show_examples(67)

## Training and Evaluation of Predictor

In [66]:
ANN_TRAIN = False
ANN_INFER = False #True
THRESHOLD_SAE = False

if ANN_TRAIN:
    assert ANN_INFER or THRESHOLD_SAE
elif ANN_INFER:
    assert THRESHOLD_SAE

In [67]:
if USE_VALIDATION:
    holdout_df = val_df
else:
    raise ValueError("Need to embed the test data")
n_heldout = len(holdout_df)

In [68]:
hypotheses = hypothesis_df["interpretation"].tolist()

In [69]:
curryann = partial(annotate_texts_with_concepts, concepts=hypotheses, cache_name=ANN_CACHE_NAME, model=ANNOTATOR_MODEL, n_workers=100)

In [70]:
sub_rng = np.random.default_rng(0)
sub_df = train_df.iloc[sub_rng.choice(len(train_df), size=n_heldout, replace=False)]

In [71]:
subset_ann_A = curryann(sub_df['headline_A'].tolist())
subset_ann_B = curryann(sub_df['headline_B'].tolist())

Found 20380 cached items; annotating 0 uncached items
Found 20380 cached items; annotating 0 uncached items


In [72]:
def display_res(m, df):
    print(f"AUC: {m['auroc']:.3f}")
    print(f"Significant hypotheses: {m['Significant'][0]}/{m['Significant'][1]} " 
          f"(p < {m['Significant'][2]:.3e})")
    pd.set_option('display.max_colwidth', None)
    display(df.sort_values(by='separation_score', ascending=False).round(4))
    pd.reset_option('display.max_colwidth')
    

In [73]:
def make_sae_delta_dict(Z_delta, sign):
    Z_delta = np.where(np.abs(Z_delta) > 1e-6, Z_delta, 0)
    if sign:
        Z_delta = np.sign(Z_delta)
    return {
        row['interpretation'] : Z_delta[:, neuron_idx]
        for neuron_idx, row in hypothesis_df.iterrows()
    }
def make_ann_delta_dict(headline_A_anns, headline_B_anns):
    return {key : headline_A_anns[key] - headline_B_anns[key] for key in headline_A_anns.keys()}

In [74]:
if ANN_TRAIN:
    train_delta_annots = make_ann_delta_dict(subset_ann_A, subset_ann_B)
    fit_labels = sub_df[label_col]
else:
    train_delta_annots = make_sae_delta_dict(train_Z_delta, sign=THRESHOLD_SAE)
    fit_labels = train_df[label_col]

_, _, train_predictor = score_hypotheses(
    hypothesis_annotations=train_delta_annots,
    y_true=fit_labels,
    classification=True,
)

Optimization terminated successfully.
         Current function value: 0.625529
         Iterations 6


## Evaluating the Predictor

In [75]:
heldout_headline_A = curryann(holdout_df["headline_A"].tolist())
heldout_headline_B = curryann(holdout_df["headline_B"].tolist())
labels_holdout = holdout_df[label_col].values

Found 20380 cached items; annotating 0 uncached items
Found 20380 cached items; annotating 0 uncached items


In [76]:
if ANN_INFER:
    eval_delta_annots = make_ann_delta_dict(heldout_headline_A, heldout_headline_B)
else:
    heldout_delta = get_delta_acts_from_df(holdout_df)
    eval_delta_annots = make_sae_delta_dict(heldout_delta, sign=THRESHOLD_SAE)
metrics, evaluation_df, _ = score_hypotheses(
    hypothesis_annotations=eval_delta_annots,
    y_true=labels_holdout,
    classification=True,
    predictor=train_predictor
)
display_res(metrics, evaluation_df)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

AUC: 0.709
Significant hypotheses: 20/20 (p < 5.000e-03)


,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence,significant,auc
Index,,,,,,,,
14,Mentions a specific time duration (minutes or seconds) in the headline (positive) versus no mention of any time duration (negative),0.2104,0.0568,2.9156,0.0000,0.0510,True,0.5211
1,Mentions a video (positive) or mentions a book/novel/fictional work (negative),0.1608,0.0000,3.7612,0.0000,0.1541,True,0.5488
10,"References a protected demographic group (e.g., race in positive samples, gender in negative samples)",0.1604,0.0000,4.6353,0.0000,0.0716,True,0.5221
6,"Contains a number written as a word (e.g., “One”) in positive samples or contains a numeric digit (e.g., “7”, “49”) in negative samples",0.1359,0.0000,5.7751,0.0000,0.0805,True,0.5206
13,Contains the word 'Response' (positive) vs contains the word 'question' (negative),0.1353,0.0000,4.6591,0.0000,0.0520,True,0.5143
12,"Mentions a controversial social issue (e.g., LGBTQ+ rights, consent, sexual violence) in positive samples or mentions a controversial environmental/energy issue (e.g., oil spills, pipelines) in negative samples",0.1207,0.0000,4.2018,0.0001,0.0579,True,0.5141
3,"Mentions photographic media (photos/images) in positive samples versus mentions video/broadcast media (videos, TV, watching) in negative samples",0.1069,0.0000,5.8125,0.0000,0.0785,True,0.5172
11,"Contains a numeric token (e.g., 51, 5, 50+)",0.1033,0.0000,2.3712,0.0000,0.1963,True,0.5388
2,Mentions a marginalized identity or social injustice (positive) or mentions an environmental/climate issue (negative),0.0888,0.0000,4.2200,0.0000,0.1590,True,0.5276
